In [4]:
!pip install ultralytics roboflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.6 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.2 MB/s eta 0:00:0000:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-s

In [5]:
from roboflow import Roboflow

# Replace with your specific API Key and Project/Version details from Roboflow
rf = Roboflow(api_key="Your API key")
project = rf.workspace("ankits-workspace-vivos").project("wagon-detection-eh2ov-evvre-ijds0")
dataset = project.version(1).download("yolov8") # Note: yolov8 format is compatible with YOLO11

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Wagon-detection-1 in yolov8:: 100%|██████████| 3906/3906 [00:00<00:00, 7330.51it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [9]:
import yaml
from ultralytics import YOLO
import os

# 1. Ensure paths are correctly pointing to the local train/val/test folders
with open("data.yaml", "r") as f:
    data = yaml.safe_load(f)

# If the folders are in the current directory (Wagon-detection-1), 
# remove any '../' prefixes.
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

with open("data.yaml", "w") as f:
    yaml.dump(data, f)

# 2. Verify files exist before training (Safety Check)
print(f"Checking for training images in: {os.path.join(os.getcwd(), data['train'])}")

# 3. Initialize and Train
# We use yolo11s (Small) for a balance of accuracy and speed
model = YOLO("yolo11n.pt")

# 3. Train with "Industry-Grade" settings
model.train(
    data="data.yaml",
    epochs=150,             # Increased epochs for convergence
    imgsz=800,              # Higher resolution = better detail for defects
    batch=8,                # Lower batch size to compensate for higher resolution/larger model
    
    # --- Motion Blur Mitigation ---
    augment=True,           # Enables standard augmentations
                   # Adds synthetic blur randomly during training
    
    # --- Optimization ---
    optimizer='AdamW',      # Often performs better than SGD for inspection tasks
    patience=100, # Stop early if accuracy stops improving
    degrees=5.0,
    name="Damage_Detector"
)

Checking for training images in: /kaggle/working/Wagon-detection-1/train/images
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Damage_Detector, nbs=64, nms=False, opset=No

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79fea1b81a60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [7]:
import os

# 1. Print the current directory
print("Current Working Directory:", os.getcwd()) 

# 2. List all files in the current directory to verify
print("Files in current directory:", os.listdir('.'))

# 3. Check if your data is inside the Wagon-detection-1 folder
if os.path.exists('Wagon-detection-1'):
    print("Found 'Wagon-detection-1' folder. Moving into it...")
    os.chdir('Wagon-detection-1')
    print("New Working Directory:", os.getcwd())
else:
    print("Could not find 'Wagon-detection-1'. Please check your file sidebar.")

Current Working Directory: /kaggle/working
Files in current directory: ['Wagon-detection-1', '.virtual_documents']
Found 'Wagon-detection-1' folder. Moving into it...
New Working Directory: /kaggle/working/Wagon-detection-1


In [10]:
!zip -r output_files.zip /kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/

  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/ (stored 0%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/confusion_matrix.png (deflated 24%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/train_batch29821.jpg (deflated 13%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/val_batch0_pred.jpg (deflated 5%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/BoxPR_curve.png (deflated 14%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/train_batch0.jpg (deflated 8%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/val_batch2_pred.jpg (deflated 8%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/BoxR_curve.png (deflated 9%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/results.csv (deflated 61%)
  adding: kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/train_batch1.jpg (d

In [11]:
from IPython.display import FileLink

# For a single file
display(FileLink(r'/kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/weights/best.pt'))

# For the zipped folder created in the previous step
display(FileLink(r'output_files.zip'))

/kaggle/working/Wagon-detection-1/runs/detect/Damage_Detector/weights/best.pt

/kaggle/working/Wagon-detection-1/output_files.zip